#Limpieza controlada (PROD)
Objetivo: eliminar las tablas externas Delta y sus datos físicos de las capas Bronze, Silver y Golden del catálogo banking_prod.

**Alcance**:
- Catálogo: banking_prod
- Esquemas: bronze, silver, golden
- Almacenamiento: cuenta adlsprojectbank12
- Contenedores: bronze, silver, golden  

No elimina: archivos de Raw, contenedores, Storage Account, Storage Credential, External Locations, catálogo, schemas, clústeres, Jobs ni notebooks.

###Parámetros

Se agregan únicamente las tablas que deseas eliminar en cada lista. Se escriben los nombres sin catálogo ni esquema. Se deja vacía una lista si no es necesario eliminar tablas de esa capa.

In [0]:
from databricks.sdk import WorkspaceClient

# Configuración del proyecto
catalogo = "banking_prod"
nameStorage = "adlsprojectbank12"

# Seleccionamos solo lo que se desea eliminar, ejm: tablas_bronze = ["cards","customers"]
tablas_bronze = []
tablas_silver = []
tablas_golden = []

# Delta Sharing: activar solo los objetos a eliminar
ELIMINAR_SHARE = False
ELIMINAR_RECIPIENT = False
share_name = "banking_analytics_share"
recipient_name = "powerbi_banking_prod"

# Seguridad general
DRY_RUN = True  # Se cambia a False para eliminar realmente

# Cliente Databricks
w = WorkspaceClient()

# Configuración de capas y contenedores
capas = {
    "bronze": {"tablas": tablas_bronze, "container": "bronze"},
    "silver": {"tablas": tablas_silver, "container": "silver"},
    "golden": {"tablas": tablas_golden, "container": "golden"}
}

print(f"Catálogo: {catalogo}")
print(f"Storage Account: {nameStorage}")
print(f"Modo simulación: {DRY_RUN}")
print(f"Eliminar Share: {ELIMINAR_SHARE} ({share_name})")
print(f"Eliminar Recipient: {ELIMINAR_RECIPIENT} ({recipient_name})")


###Eliminar tablas seleccionadas

In [0]:
for esquema, config in capas.items():
    print(f"\\n===== {esquema.upper()} =====")

    for nombre_tabla in config["tablas"]:
        nombre_completo = f"{catalogo}.{esquema}.{nombre_tabla}"
        print(f"Tabla: {nombre_completo}")

        try:
            tabla = w.tables.get(full_name=nombre_completo)

            tipo = (
                tabla.table_type.value
                if hasattr(tabla.table_type, "value")
                else str(tabla.table_type)
            )
            formato = (
                tabla.data_source_format.value
                if hasattr(tabla.data_source_format, "value")
                else str(tabla.data_source_format)
            )
            ruta = tabla.storage_location or ""

            ruta_esperada = (
                f"abfss://{config['container']}@{nameStorage}."
                f"dfs.core.windows.net/banking_prod/"
            )

            if tipo.upper() != "EXTERNAL":
                print("OMITIDA: no es una tabla externa.")
                continue

            if formato.upper() != "DELTA":
                print("OMITIDA: no es una tabla Delta.")
                continue

            if not ruta.startswith(ruta_esperada):
                print(f"OMITIDA: ruta fuera del alcance permitido: {ruta}")
                continue

            if DRY_RUN:
                print(f"SIMULACIÓN: se eliminaría {nombre_completo}")
                print(f"SIMULACIÓN: también se eliminaría {ruta}")
            else:
                # Primero se elimina el objeto de Unity Catalog
                w.tables.delete(full_name=nombre_completo)

                # Después se elimina su carpeta física en ADLS
                dbutils.fs.rm(ruta, True)

                print(f"ELIMINADA: {nombre_completo}")

        except Exception as e:
            # Si la tabla no existe o falla la operación, informa y continúa
            print(f"ERROR con {nombre_completo}: {e}")

print("\\nProceso finalizado.")

###Eliminar objetos de delta sharing

In [0]:
# Eliminación selectiva de Delta Sharing

if not ELIMINAR_SHARE and not ELIMINAR_RECIPIENT:
    print("Delta Sharing: no se seleccionó ningún objeto para eliminar.")
else:
    # Primero Share y después Recipient, cuando ambas opciones están activadas
    if ELIMINAR_SHARE:
        try:
            if DRY_RUN:
                print(f"SIMULACIÓN: se eliminaría el Share '{share_name}'.")
            else:
                w.shares.delete(name=share_name)
                print(f"ELIMINADO: Share '{share_name}'.")
        except Exception as e:
            print(f"ERROR al eliminar Share '{share_name}': {e}")

    if ELIMINAR_RECIPIENT:
        try:
            if DRY_RUN:
                print(f"SIMULACIÓN: se eliminaría el Recipient '{recipient_name}'.")
            else:
                w.recipients.delete(name=recipient_name)
                print(f"ELIMINADO: Recipient '{recipient_name}'.")
        except Exception as e:
            print(f"ERROR al eliminar Recipient '{recipient_name}': {e}")

print("Proceso de Delta Sharing finalizado.")
